# Etapa 5d — Algoritmo final en una función y máscaras de desarrollo

El algoritmo adoptado en `05b`, ampliado en la ronda de mejoras de `05e` (cada filtro también en cada escala por separado, 26 mapas; plan, desvío 14.6) y entrenado con un tercio de píxeles de vaso de cada grosor (`05f`, desvío 14.8) queda en `src/pipeline.py` (`segment_image`; la referencia clásica en `segment_reference`), con un script para imágenes nuevas (`scripts/segmentar.py`). Este cuaderno:

1. **Máscaras de desarrollo con validación cruzada:** cada imagen se segmenta con los pesos y el umbral elegidos sin su persona (las 5 vueltas de siempre) y se guardan en `results/cache/mascaras_cv/`: el algoritmo final y la referencia, cada uno con fracción fija y con valor fijo (desvío 14.7). Las usan las etapas 6 y 7.
2. **Verificación:** Matthews medio de esas máscaras contra el de `05f` (B, entrenamiento equilibrado: 0,7255 con la fracción y 0,7302 con el valor fijo) y `05b` (detector de líneas: 0,7022 con la fracción y 0,7040 con el valor fijo). Tiene que dar lo mismo.
3. **Pesos, fracción y valor fijo con las 30 imágenes de desarrollo**, guardados en `models/pesos_combinacion.json` para el script. Se confirman al congelar (etapa 9).


In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.pipeline import choose_fraction, choose_value, fit_weights, save_weights
from src.split import person_folds

ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))

In [2]:
def pass1(image_id):
    from src.pipeline import training_pixels
    return image_id, training_pixels(image_id)

t = time.time()
samples = dict(map_images(pass1, ids, workers=4))
fits = {}
for k in range(5):
    tr = list(folds.index[folds != k])
    w = fit_weights(samples, tr)
    fits[k] = {"w": w, "combinación": (choose_fraction(samples, w, tr), choose_value(samples, w, tr)),
               "referencia": (choose_fraction(samples, None, tr), choose_value(samples, None, tr))}
print(f"Pesos y umbrales por vuelta listos en {time.time() - t:.0f} s")
pd.DataFrame({(m, r): [fits[k][m][j] for k in range(5)] for m in ("combinación", "referencia")
              for j, r in enumerate(("fracción", "valor fijo"))}).round(4)

Pesos y umbrales por vuelta listos en 406 s


combinación            referencia           
     fracción valor fijo   fracción valor fijo
0        0.10     0.8713       0.09     0.9710
1        0.10     0.8566       0.09     0.9761
2        0.09     0.8567       0.09     0.9284
3        0.09     0.8567       0.09     0.9508
4        0.09     0.8617       0.09     0.9451

In [3]:
def pass2(image_id):
    from src.data import load_fov, load_image, load_vessels
    from src.metrics import matthews
    from src.pipeline import compute_features, save_dev_masks, segment_image, segment_reference
    rgb = load_image(image_id)
    f, _ = compute_features(rgb)
    fit = fits[int(folds[image_id])]
    (cf, cv), (rf, rv) = fit["combinación"], fit["referencia"]
    masks = {"combinación": segment_image(rgb, fit["w"], fraction=cf, features=f),
             "referencia": segment_reference(rgb, fraction=rf, features=f),
             "combinación valor fijo": segment_image(rgb, fit["w"], fraction=None, value=cv, features=f),
             "referencia valor fijo": segment_reference(rgb, fraction=None, value=rv, features=f)}
    save_dev_masks(image_id, masks)
    truth, fov = load_vessels(image_id), load_fov(image_id)
    return {"image": image_id, **{k: matthews(m, truth, fov) for k, m in masks.items()}}

t = time.time()
M = pd.DataFrame(map_images(pass2, ids, workers=4)).set_index("image")
print(f"Máscaras guardadas en {time.time() - t:.0f} s")
print(M.mean().round(4).to_string())

Máscaras guardadas en 489 s
combinación               0.7255
referencia                0.7022
combinación valor fijo    0.7302
referencia valor fijo     0.7040


In [4]:
final = fit_weights(samples, ids)
final_fraction, final_value = choose_fraction(samples, final, ids), choose_value(samples, final, ids)
save_weights(final, fraction=final_fraction, value=final_value,
             note="Ajustados con las 30 imágenes de desarrollo (05d, 31/10); se confirman al congelar.")
print(f"Con las 30: fracción {final_fraction} (config.SEGMENT_FRACTION = {C.SEGMENT_FRACTION}), valor fijo {final_value:.4f}")
print(pd.Series(final, index=["ordenada"] + C.SEGMENT_FEATURES).round(3).to_string())

Con las 30: fracción 0.09 (config.SEGMENT_FRACTION = 0.09), valor fijo 0.8573
ordenada                -1.561
verde corregido          0.700
sombrero de copa         0.573
detector de líneas      -2.092
filtro adaptado         -0.147
Gabor                    0.811
Frangi                   1.680
detector de líneas 2    -0.075
filtro adaptado 2       -0.017
Gabor 2                 -0.456
Frangi 2                -0.559
detector de líneas 4     0.183
filtro adaptado 4        0.277
Gabor 4                 -0.055
Frangi 4                 0.612
detector de líneas 8     0.806
filtro adaptado 8        1.021
Gabor 8                 -1.207
Frangi 8                -1.578
detector de líneas 16    1.399
filtro adaptado 16      -0.668
Gabor 16                 0.340
Frangi 16               -0.986
detector de líneas 24    3.576
filtro adaptado 24      -0.555
Gabor 24                -0.277
Frangi 24               -1.131
